In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

# 🧱 W13-D4 概念实验：Vision Agent vs LangChat Agent 的三个结构差异

主线 md 结论：Vision Agent 和 LangChat Agent 是同一物种（都是数字员工），差异在三件事——
1. **时钟**：谁触发（请求驱动 vs 班次驱动）
2. **证据**：凭什么推理（结构化事实 vs 统计量，后者必须携带置信度）
3. **失败模式**：错了怎么办（fail-closed 拒绝 vs 永不停机降级）

本 notebook 用三个小实验分别验证这三条。数据全部模拟（无联网、无真实模型），只验证**概念结构**。

## 实验 1：时钟 —— 两种 Agent 的 24 小时动作分布

LangChat Agent（事务型）：动作由用户请求触发，泊松到达，无请求即空闲。
Vision Agent（观测型）：动作由时间驱动——周期检测 tick + 每日 06:30 一次日报 burst。空闲即事故。

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

hours = np.linspace(0, 24, 24 * 60)  # 1 分钟分辨率

# --- LangChat Agent：营业时间 9-21 点的泊松请求（租户问询） ---
lam = 0.15  # 每分钟期望请求数
req_events = rng.poisson(lam, size=len(hours))
txn_times = hours[req_events > 0]

# --- Vision Agent：每 5 分钟一个检测 tick + 06:30 日报 ---
tick_times = np.arange(0.5, 24, 5.0 / 60)  # 每 5 分钟
report_times = np.array([6.5])

fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
axes[0].scatter(txn_times, np.ones_like(txn_times), s=18, color="#3b82f6", marker="|", linewidth=2)
axes[0].set_yticks([]); axes[0].set_title("LangChat Agent（事务型）：请求驱动 —— 空闲是常态，错误可拒绝（fail-closed）")
axes[1].scatter(tick_times, np.ones_like(tick_times), s=18, color="#94a3b8", marker="|", linewidth=1.5)
axes[1].scatter(report_times, [1], s=260, color="#ef4444", marker="*", zorder=3, label="06:30 日报（唯一『说话』时刻）")
axes[1].set_yticks([]); axes[1].legend(loc="upper right"); axes[1].set_title("Vision Agent（观测型）：时间驱动 —— 观测永不停机，到点必须报告")
axes[1].set_xlabel("小时")
axes[0].set_xlim(0, 24)
plt.tight_layout(); plt.savefig("/tmp/w13d4_clock.png", dpi=110); plt.show()

print(f"事务 Agent 全天动作数: {len(txn_times)}（随机，{len(txn_times)==0 and '可以为 0' or '也可以很多'}）")
print(f"观测 Agent 全天动作数: {len(tick_times)} 检测 tick + {len(report_times)} 日报（确定性，停=事故）")

**观察**：上图的散点是随机的（没有请求就没有动作），下图的 tick 是等间隔的网格加一颗星。
这解释了为什么两者的失败模式不同——**事务系统停机损失的是响应，观测系统停机损失的是证据**。

## 实验 2：证据 —— 单点数字 vs 携带置信度的统计量

模拟 B1 收银区排队（Day3 的 Little's Law 场景）：每 5 分钟一次采样估计队长 L 与到达率 λ，
检测带随机噪声 + 系统性漏检（遮挡，Day3 结论：噪声平均可消，偏差平均只会固化）。

对比两种"话术"：
- **automation_report**：单点数字 + 阈值规则（>10 分钟 → "拥堵"）
- **agent_report**：bootstrap 置信区间 + 证据强度分支（区间太宽 → 主动说"数据不足"）

In [ ]:
# --- 模拟一天排队数据：真实值 → 检测值 ---
t = np.arange(11, 21, 5/60)                       # 11:00-20:00，每 5 分钟一个采样
lunch = np.exp(-0.5*((t-12.5)/1.2)**2)            # 午高峰
dinner = 0.8*np.exp(-0.5*((t-18.5)/1.3)**2)       # 晚高峰
true_W = 2 + 14*lunch + 8*dinner                  # 真实平均等待（分钟），峰值 ~16

noise = rng.normal(0, 2.0, len(t))                # 随机噪声（可平均消）
bias = -0.15*true_W                                # 系统性遮挡漏检 -15%（平均固化）
det_W = true_W + noise + bias                      # 检测得到的等待估计（Little's Law 之后）

# --- bootstrap：对一天采样重抽样，得到日峰值指标的置信区间 ---
boot_peaks = []
for _ in range(3000):
    sample = rng.choice(det_W, size=len(det_W), replace=True)
    # 每 30 分钟滑窗均值取最大 ≈ "高峰时段平均等待"
    win = 6
    peaks = [sample[i:i+win].mean() for i in range(0, len(sample)-win+1, win)]
    boot_peaks.append(max(peaks))
boot_peaks = np.array(boot_peaks)
ci_lo, ci_hi = np.percentile(boot_peaks, [2.5, 97.5])
point_est = boot_peaks.mean()

def automation_report(est):
    """规则触发：单点数字 + 硬阈值"""
    return f"自动化日报：高峰平均等待 {est:.1f} 分钟 → {'告警：拥堵，建议加开收银' if est > 10 else '正常'}"

def agent_report(est, lo, hi, n_samples):
    """证据推理：区间 + 证据强度分支（LangChat Agent 不需要这个，因为它的事实来自可信 API）"""
    width = hi - lo
    if width > 6 or n_samples < 20:   # 证据不足 → 降级声明，而非硬给结论
        return (f"数字督导日报：高峰等待 {est:.1f} 分钟（95% CI [{lo:.1f}, {hi:.1f}]，证据弱）\n"
                f"  → 数据不足以支撑排班决策，建议人工复核 {n_samples} 个采样点")
    return (f"数字督导日报：高峰等待 {est:.1f} 分钟（95% CI [{lo:.1f}, {hi:.1f}]，样本 {n_samples}）\n"
            f"  → {'超过 10 分钟服务线，建议 11:30-13:00 加开 2 个收银台' if lo > 10 else '在服务线内，无需干预'}")

print(automation_report(point_est)); print()
print(agent_report(point_est, ci_lo, ci_hi, len(det_W)))
print(f"\n真实峰值（上帝视角）: {true_W.max():.1f} 分钟  |  检测均值偏低原因: -15% 系统性遮挡偏差")

In [ ]:
# --- 证据衰减演示：采样点减少 → 区间变宽 → agent 主动降级 ---
def daily_peak(arr, win=6):
    """与上面主 bootstrap 相同口径：滑窗均值取最大（Metric 口径保持一致，Day3 的教训）"""
    return max(arr[i:i+win].mean() for i in range(0, max(1, len(arr)-win+1)))

for n in [108, 24, 8]:
    idx = np.linspace(0, len(det_W)-1, n).astype(int)   # 抽稀采样 = 模拟摄像头部分离线
    sub = det_W[idx]
    bp = [daily_peak(sub[rng.integers(0, n, n)]) for _ in range(1500)]
    lo_, hi_ = np.percentile(bp, [2.5, 97.5])
    print(f"采样点 {n:3d} → CI 宽度 {hi_-lo_:5.1f} 分钟 → {'证据充足' if hi_-lo_ <= 6 else '证据不足，日报降级为「建议人工复核」'}")


**观察**：同一批数据，自动化报告给出一个"看起来精确"的数字；agent 报告给出区间和证据强度分支。
**采样越稀（摄像头在线率低），区间越宽，agent 越应该主动说"我不知道"**——这就是"携带置信度说话"。
而 LangChat Agent 查 MI ERP 合同余额时不需要这套——它的证据来自可信 API。这就是证据维度的物种内差异。

## 实验 3：失败模式 —— fail-closed vs 永不停机降级

30 天模拟：每天证据质量随机波动（个别天摄像头离线，采样极少）。两种策略：
- **fail-closed**（事务系统美德）：证据不足 → 不出报告
- **降级**（观测系统美德）：照常出报告，但标注"证据弱，建议复核"

统计：盲区天数（真实高峰 >10 分钟但管理层没被告知）vs 人工复核工作量。

In [ ]:
days = 30
true_peaks = rng.normal(11.5, 3.0, days)          # 真实高峰等待
true_peaks = np.clip(true_peaks, 3, 20)
camera_up = rng.random(days) > 0.15               # 15% 概率摄像头部分离线
n_samples = np.where(camera_up, rng.integers(60, 110, days), rng.integers(4, 12, days))

def estimate(n, true_peak):
    """给定采样数和真实值，返回 (点估计, CI宽度) —— 采样越少越不准越宽"""
    det = true_peak*(1-0.15) + rng.normal(0, 2.0, n)
    return det.mean(), 12.0/np.sqrt(n) + 1.0

blind_failclosed, review_failclosed = 0, 0
blind_degrade, review_degrade = 0, 0
for d in range(days):
    est, width = estimate(n_samples[d], true_peaks[d])
    strong = width <= 6
    alarm_warranted = true_peaks[d] > 10
    # 策略 A：fail-closed —— 证据不足不出报告
    if strong:
        pass_informed = est > 9   # 强证据时正常报告
        if alarm_warranted and not pass_informed: blind_failclosed += 1
    else:
        if alarm_warranted: blind_failclosed += 1   # 盲区：有事件但没报告
    # 策略 B：降级 —— 永远出报告，弱证据标注复核
    if not strong:
        review_degrade += 1
        if not alarm_warranted: pass               # 复核了但没事（可接受的成本）
    else:
        if est > 9 and not alarm_warranted: blind_degrade += 1  # 强证据但估计错（两种策略共有）

print(f"策略 A（fail-closed 不确定性）: 盲区 {blind_failclosed}/{days} 天（有拥堵管理层不知情）")
print(f"策略 B（永不停机 + 降级标注）: 盲区 {blind_degrade}/{days} 天（仅强证据误判才盲），人工复核 {review_degrade} 天")

fig, ax = plt.subplots(figsize=(12, 3.8))
colors_a = np.where(n_samples<20, "#f59e0b", "#3b82f6")
ax.scatter(range(days), true_peaks, c=colors_a, s=np.where(n_samples<20, 220, 60), alpha=0.75)
ax.axhline(10, color="#ef4444", ls="--", label="10 分钟服务线")
for d in range(days):
    if n_samples[d] < 20:
        ax.annotate("离线\nfail-closed 盲区", (d, true_peaks[d]), fontsize=7,
                    xytext=(0, 18), textcoords="offset points", ha="center", color="#b45309")
ax.set_title("30 天模拟：橙大点 = 证据弱日（fail-closed 直接变成盲区；降级策略照常报告+标注复核）")
ax.set_xlabel("天"); ax.set_ylabel("真实高峰等待（分钟）"); ax.legend()
plt.tight_layout(); plt.savefig("/tmp/w13d4_failmode.png", dpi=110); plt.show()

## 综合：三个差异一张表

| 维度 | LangChat Agent | Vision Agent | 实验验证 |
|---|---|---|---|
| 时钟 | 请求驱动（泊松到达，空闲正常） | 时间驱动（tick 网格+定时报告，停机=事故） | 实验 1 |
| 证据 | API 结构化事实（可信任） | 检测统计量（噪声+偏差，须 bootstrap 带置信度） | 实验 2 |
| 失败模式 | fail-closed 拒绝执行 | 永不停机 + 降级标注（fail-closed 会制造盲区） | 实验 3 |

**结论（与主线 md 呼应但不重复）**：三组实验都说明——差异不在"要不要 Agent 框架"（治理、Skill、数字员工机制完全可复用），而在**话术与证据学**：日报必须携带置信区间，证据不足必须降级而不是沉默，观测永远不能停。这正是推理层（L5 Skill）的设计输入。